In [4]:
import sys
print(sys.executable)


c:\Program Files\Python313\python.exe


In [5]:
import xgboost

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# 1️⃣ Load dataset
df = pd.read_csv('../ligand_mutation_with_labels_heuristic.csv')

# 2️⃣ Identify columns
target = 'DockingScore_label'
categorical_cols = ['LigandName', 'Mutation', 'LigandCID']
fingerprint_cols = [col for col in df.columns if col.startswith('FP_')]
numerical_cols = [col for col in df.columns if col not in categorical_cols + fingerprint_cols + [target]]

# 3️⃣ Handle missing values
# Numerical: median
num_imputer = SimpleImputer(strategy='median')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# Fingerprints: fill NaN with 0
df[fingerprint_cols] = df[fingerprint_cols].fillna(0)

# Categorical: fill NaN with 'Unknown'
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# Target: drop rows where DockingScore_label is NaN
df = df[~df[target].isna()]

# 4️⃣ Encode categorical features
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# 5️⃣ Scale numerical features
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# 6️⃣ Prepare features and target
X = df[numerical_cols + fingerprint_cols + categorical_cols]
y = df[target]

# 7️⃣ Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8️⃣ Train XGBoost regressor
model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

# 9️⃣ Evaluate
y_pred = model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

# 🔟 Save the model for app integration
joblib.dump(model, 'xgb_drug_model.pkl')
print("Model saved as xgb_drug_model.pkl")


MSE: 2.3182022504180132e-08
R2: 0.9999999854796896
Model saved as xgb_drug_model.pkl


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# 1️⃣ Load dataset
df = pd.read_csv('../ligand_mutation_with_labels_heuristic.csv')

# 2️⃣ Identify columns
target = 'DockingScore_label'
categorical_cols = ['LigandName', 'Mutation', 'LigandCID']
fingerprint_cols = [col for col in df.columns if col.startswith('FP_')]
numerical_cols = [col for col in df.columns if col not in categorical_cols + fingerprint_cols + [target]]

# 3️⃣ Handle missing values
num_imputer = SimpleImputer(strategy='median')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# Fingerprints: fill NaN with 0
df[fingerprint_cols] = df[fingerprint_cols].fillna(0)

# Categorical: fill NaN with 'Unknown'
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# Target: drop rows where DockingScore_label is NaN
df = df[~df[target].isna()]

# 4️⃣ Encode categorical features and save encoders
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    joblib.dump(le, f'label_encoder_{col}.pkl')

# 5️⃣ Scale numerical features and save scaler
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
joblib.dump(scaler, 'scaler.pkl')

# 6️⃣ Save numerical imputer
joblib.dump(num_imputer, 'num_imputer.pkl')

# 7️⃣ Prepare features and target
X = df[numerical_cols + fingerprint_cols + categorical_cols]
y = df[target]

# 8️⃣ Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 9️⃣ Train XGBoost regressor
model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

# 🔟 Evaluate
y_pred = model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

# 1️⃣1️⃣ Save the model
joblib.dump(model, 'xgb_drug_model.pkl')
print("Model and preprocessing objects saved for app integration.")

MSE: 2.3182022504180132e-08
R2: 0.9999999854796896
Model and preprocessing objects saved for app integration.
